In [75]:
!pip install -q pydantic-settings langchain-text-splitters langchain-community langchain-huggingface faiss-cpu

In [88]:
from warnings import filterwarnings
filterwarnings(action="ignore")
from ProjectConfiguration.SecretKeys import ProjectConfig

import os
import bs4
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

os.environ["GROQ_API_KEY"] = ProjectConfig.groq_api_key

web_path = "https://medium.com/@spaw.co/best-website|s-to-practice-web-scraping-9df5d4df4d1"

In [62]:
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0)

llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000220B8920F50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000220B8921D90>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [33]:
llm.invoke("Hello, my name is Eddy wby?").content

"Nice to meet you, Eddy! I'm just a language model, I don't have a personal name, but you can call me Assistant or AI if you'd like. I'm here to help with any questions or topics you'd like to discuss. How's your day going so far?"

In [119]:
def LoadRetriever(web_path:str, llm):
    webloader = WebBaseLoader(
                          web_path=web_path,
                          # bs_kwargs={"parse_only":bs4.SoupStrainer(["p", "h1"])}
                         )
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,
                                                   chunk_overlap=100,
                                                   length_function=len)
    # Step 1: Load the document
    loaded_doc = webloader.load()
    
    # Step 2: Split the document
    splitted_doc = text_splitter.split_documents(loaded_doc)
    
    print(f"Document splitted into: {len(splitted_doc)} document")
    
    # Step 3: Text Embedding 
    model_name = "sentence-transformers/all-mpnet-base-v2"
    model_kwargs = {'device': 'cpu'}
    encode_kwargs = {'normalize_embeddings': True}
    embedding = HuggingFaceEmbeddings(
                model_name=model_name,
                model_kwargs=model_kwargs,
                encode_kwargs=encode_kwargs
            )
    
    # Step 4: Embed and Store Text
    vector_embeddings = FAISS.from_documents(documents=splitted_doc, embedding=embedding)
    
    # Step 5: Create retriever chain
    vectore_retriver = vector_embeddings.as_retriever(search_type="similarity",
                                                      search_kwargs={"k":5})
    
    print("Retriever created")

    prompt = ChatPromptTemplate.from_messages([
        ("system", ("You are a helpful and concise AI assistant." 
                    "Use the provided context to answer the user's question." 
                    "If the answer is not in the context, respond with 'I don't know'.")),
        ("user", "Context:\n{context}\n\nQuestion:\n{question}")
    ])

    qa_retriever = RetrievalQA.from_chain_type(llm=llm,
                                               retriever=vectore_retriver,
                                               chain_type="stuff", 
                                               chain_type_kwargs={"prompt":prompt})

    return qa_retriever

In [120]:
retriever = LoadRetriever(web_path=web_path, llm=llm)

Document splitted into: 13 document
Retriever created


In [122]:
retriever.invoke({"query":"Define Webscraping"})["result"]

'Web scraping, commonly referred to as web harvesting or web data extraction, is a technique used to extract vast amounts of data from websites quickly. This data can be saved to your computer in a format of your choice, such as CSV or Excel, to be later analyzed or utilized.'